#  Lab 7 – Hashing


---

## Bài 1 – Rolling Hash & Rabin–Karp

### 1.1. Bài toán

**Cho:**
- Chuỗi text `T` (ví dụ: `"ababcabcababd"`).
- Chuỗi pattern `P` (ví dụ: `"abc"`).

**Yêu cầu:**
- Viết 2 hàm:
  - `brute_force_search(text, pattern)` – trả về index đầu tiên P xuất hiện trong T (hoặc -1).
  - `rolling_hash_search(text, pattern)` – dùng rolling hash đơn giản để tìm.
- So sánh kết quả hai hàm, in ra index tìm được.

### 1.2. Hàm Brute-Force

In [1]:
def brute_force_search(text, pattern):
    
    n, m = len(text), len(pattern)
    if m == 0 or m > n:
        return -1

    for i in range(n - m + 1):
        if text[i:i+m] == pattern:
            return i

    return -1

### 1.3. Hàm Hash đơn giản

In [2]:
def simple_hash(s):
    
    return sum(ord(c) for c in s)

### 1.4. Rolling Hash Search

**Ý tưởng cửa sổ trượt:**
- Tính hash ban đầu cho cửa sổ đầu tiên.
- Khi trượt sang phải 1 vị trí: **trừ** ký tự rời khỏi cửa sổ, **cộng** ký tự mới vào cửa sổ.
- Nếu hash trùng → kiểm tra lại bằng so chuỗi (tránh false positive).

In [3]:
def rolling_hash_search(text, pattern):
    
    n, m = len(text), len(pattern)
    if m == 0 or m > n:
        return -1

    # 1. Tính hash pattern
    hash_p = simple_hash(pattern)

    # 2. Tính hash cửa sổ đầu tiên của text
    window = text[:m]
    hash_w = simple_hash(window)

    
    for i in range(n - m + 1):
        #  trùng → so chuỗi
        if hash_w == hash_p:
            if text[i:i+m] == pattern:
                return i

        # Cập nhật hash_w 
        if i < n - m:
            # Cập nhật rolling hash:
            # - trừ ký tự text[i] (rời khỏi cửa sổ)
            # - cộng ký tự text[i + m] (vào cửa sổ)
            hash_w -= ord(text[i])
            hash_w += ord(text[i + m])

    return -1

### 1.5. Test Rolling Hash

In [4]:
def test_rolling_hash():
    text = "ababcabcababd"
    pattern = "abc"

    i1 = brute_force_search(text, pattern)
    i2 = rolling_hash_search(text, pattern)

    print("Text   :", text)
    print("Pattern:", pattern)
    print("Brute force index :", i1)
    print("Rolling hash index:", i2)

if __name__ == "__main__":
    test_rolling_hash()

Text   : ababcabcababd
Pattern: abc
Brute force index : 2
Rolling hash index: 2


---

## Bài 2 – Group Anagrams

### 2.1. Bài toán

Cho mảng `words` gồm các string.
- Nhóm các từ là **anagram** của nhau.
- Anagram: cùng ký tự, khác thứ tự.

**Ví dụ:**
- Input: `["eat","tea","tan","ate","nat","bat"]`
- Output: `[["eat","tea","ate"],["tan","nat"],["bat"]]`

### 2.2. Ý tưởng

Cách đơn giản: **sort từng từ** làm key:
- `"eat"`, `"tea"`, `"ate"` → `"aet"`
- `"tan"`, `"nat"` → `"ant"`

Dùng `dict`:
- **key**: chuỗi đã sort.
- **value**: list các từ thuộc nhóm.

### 2.3. Code

In [5]:
from collections import defaultdict

def group_anagrams(words):
    
    groups = defaultdict(list)

    for w in words:
        # Tạo key bằng cách sort w
        key = ''.join(sorted(w))
        # Thêm w vào groups[key]
        groups[key].append(w)

    # Trả về list các group
    return list(groups.values())

### 2.4. Test Group Anagrams

In [6]:
def test_group_anagrams():
    words = ["eat", "tea", "tan", "ate", "nat", "bat"]
    result = group_anagrams(words)
    print("Input :", words)
    print("Output groups:")
    for group in result:
        print("  ", group)

if __name__ == "__main__":
    test_group_anagrams()

Input : ['eat', 'tea', 'tan', 'ate', 'nat', 'bat']
Output groups:
   ['eat', 'tea', 'ate']
   ['tan', 'nat']
   ['bat']


---

## Bài 3 – Longest Consecutive Sequence

### 3.1. Bài toán

Cho mảng `nums` không được sort trước.
- Tìm **độ dài dãy số liên tiếp dài nhất**.

**Ví dụ:**
- `[100, 4, 200, 1, 3, 2]` → dãy `[1,2,3,4]` → **4**
- `[0,3,7,2,5,8,4,6,0,1]` → dãy `[0,1,2,3,4,5,6,7,8]` → **9**

**Yêu cầu:** O(n).

### 3.2. Ý tưởng Hash Set

Dùng `set` để kiểm tra tồn tại O(1):
1. Đưa toàn bộ `nums` vào `set`.
2. Với mỗi `x` trong set, **chỉ bắt đầu đếm** khi `x` là **đầu dãy**: `x-1` không nằm trong set.
3. Từ `x`, tăng dần `x+1, x+2,...` cho đến khi không còn trong set, đếm độ dài.

Mỗi phần tử là đầu dãy nhiều nhất một lần → **O(n)**.

### 3.3. Code

In [7]:
def longest_consecutive(nums):
    
    if not nums:
        return 0

    num_set = set(nums)
    best = 0

    for x in num_set:
        # Chỉ bắt đầu nếu x là "đầu dãy" (x-1 không trong set)
        if x - 1 not in num_set:
            length = 1
            cur = x
            while cur + 1 in num_set:
                cur += 1
                length += 1
            if length > best:
                best = length

    return best

### 3.4. Test Longest Consecutive

In [8]:
def test_longest_consecutive():
    nums1 = [100, 4, 200, 1, 3, 2]
    nums2 = [0, 3, 7, 2, 5, 8, 4, 6, 0, 1]

    print("nums1:", nums1)
    print("Longest consecutive:", longest_consecutive(nums1))  # 4

    print("nums2:", nums2)
    print("Longest consecutive:", longest_consecutive(nums2))  # 9

if __name__ == "__main__":
    test_longest_consecutive()

nums1: [100, 4, 200, 1, 3, 2]
Longest consecutive: 4
nums2: [0, 3, 7, 2, 5, 8, 4, 6, 0, 1]
Longest consecutive: 9


---

## Bài 4 – Subarray Sum = k

### 4.1. Bài toán

**Cho:**
- Mảng `nums` (có thể âm, dương).
- Số nguyên `k`.

**Yêu cầu:**
- Đếm số subarray liên tiếp có tổng bằng `k`.

**Ví dụ:**
- `nums = [1,1,1]`, `k = 2` → có **2** subarray: `(0..1)`, `(1..2)`.
- `nums = [1,2,3]`, `k = 3` → có **2** subarray: `[1,2]`, `[3]`.

**Mục tiêu:** O(n).

### 4.2. Ý tưởng Prefix Sum + Hash Map

Gọi `prefix[i]` = tổng từ index `0..i`.

Tổng subarray `(l..r)` = `prefix[r] - prefix[l-1]`.

Muốn: `prefix[r] - prefix[l-1] = k` → `prefix[l-1] = prefix[r] - k`.

Duyệt từ trái sang phải:
- `current_sum` = prefix tại vị trí hiện tại.
- Cần biết trước đó có bao nhiêu prefix = `current_sum - k`.
- Dùng `dict prefix_count` lưu: prefix value → số lần xuất hiện.
- Mỗi bước cập nhật `result += prefix_count[current_sum - k]`.

### 4.3. Code

In [9]:
from collections import defaultdict

def subarray_sum(nums, k):
    
    prefix_count = defaultdict(int)
    prefix_count[0] = 1  # prefix sum = 0 xuất hiện 1 lần trước khi duyệt

    current_sum = 0
    result = 0

    for x in nums:
        current_sum += x

        # Tính cần_prefix = current_sum - k
        needed = current_sum - k
        # Nếu cần_prefix đã xuất hiện, cộng số lần vào result
        if needed in prefix_count:
            result += prefix_count[needed]
        # Cập nhật prefix_count[current_sum]
        prefix_count[current_sum] += 1

    return result

### 4.4. Test Subarray Sum = k

In [10]:
def test_subarray_sum():
    nums1 = [1, 1, 1]
    k1 = 2
    print("nums1:", nums1, "k =", k1)
    print("Số subarray sum = k:", subarray_sum(nums1, k1))  # 2

    nums2 = [1, 2, 3]
    k2 = 3
    print("\nnums2:", nums2, "k =", k2)
    print("Số subarray sum = k:", subarray_sum(nums2, k2))  # 2

if __name__ == "__main__":
    test_subarray_sum()

nums1: [1, 1, 1] k = 2
Số subarray sum = k: 2

nums2: [1, 2, 3] k = 3
Số subarray sum = k: 2
